# Ray RLlib: DQN on CartPole-v1

Project path: `projects/cartpole-dqn`

**Off-policy** discrete control with a prioritized replay buffer — companion step 2 in the blueprint ladder.

**Setup (once)** from the repository root:

```bash
python -m venv .venv
source .venv/bin/activate
pip install -r projects/cartpole-dqn/requirements.txt
```

Select that kernel, then run the cell below. Script twin: `python projects/cartpole-dqn/train_cartpole_dqn.py`.

Expect `episode_return_mean` to climb from ~20 toward **100–250+** over ~10 iterations (~1–2 min).

In [ ]:
import warnings
from typing import Any

warnings.filterwarnings(
    "ignore",
    message=r".*RLModule\(config=\[RLModuleConfig object\]\).*",
    category=DeprecationWarning,
)

from ray.rllib.algorithms.dqn.dqn import DQNConfig
from ray.rllib.core.rl_module.default_model_config import DefaultModelConfig


def episode_return_mean(result: dict[str, Any]) -> float | None:
    env_runners = result.get("env_runners") or {}
    value = env_runners.get("episode_return_mean")
    return float(value) if value is not None else None


config = (
    DQNConfig()
    .environment("CartPole-v1")
    .env_runners(num_env_runners=1)
    .training(
        replay_buffer_config={
            "type": "PrioritizedEpisodeReplayBuffer",
            "capacity": 60_000,
            "alpha": 0.5,
            "beta": 0.5,
        },
        num_steps_sampled_before_learning_starts=1_000,
    )
    .rl_module(model_config=DefaultModelConfig(fcnet_hiddens=[64, 64]))
    .evaluation(evaluation_num_env_runners=1)
    .debugging(log_level="ERROR")
)

algo = config.build_algo()
try:
    for i in range(1, 11):
        result = algo.train()
        ret = episode_return_mean(result)
        steps = result.get("num_env_steps_sampled_lifetime")
        if ret is not None:
            print(f"iter={i}  episode_return_mean={ret:.1f}  env_steps={steps}")
        else:
            print(f"iter={i}  env_steps={steps}")

    eval_result = algo.evaluate()
    eval_ret = episode_return_mean(eval_result)
    print(
        f"evaluate  episode_return_mean={eval_ret:.1f}"
        if eval_ret is not None
        else "evaluate  (no episode_return_mean)"
    )
finally:
    algo.stop()

## What this teaches

| Idea | In this notebook |
| --- | --- |
| Off-policy learning | DQN reuses past transitions from a replay buffer |
| Discrete control | CartPole left/right actions |
| Same API stack as Taxi | `env_runners`, `DefaultModelConfig`, `build_algo()`, `algo.stop()` |

Next: [Pendulum SAC](../pendulum-sac/pendulum_sac.ipynb) · [Project README](README.md)